# Generate Synthetic DS1

This notebook generates the public synthetic input files used by the Paper I analysis notebook.

The purpose is to create a public synthetic replacement for the original proprietary Paper I D1 dataset while preserving the analysis-ready structure used by the methodology.

The notebook produces two repository data products:

- `data/DS1.xlsx`: synthetic measured dataset with chemistry, categorical labels, measured tensile properties, and ML-ready one-hot encoded categorical columns
- `data/NaMo_syntetic.xlsx`: synthetic NaMo estimates stored separately with the same row index

## 1. Repository outputs and generation settings

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

REPO_DIR = Path.cwd().resolve()
OUTPUT_DIR = REPO_DIR / "data"
OUTPUT_DIR.mkdir(exist_ok=True)

DS1_FILE = OUTPUT_DIR / "DS1.xlsx"
NAMO_FILE = OUTPUT_DIR / "NaMo_syntetic.xlsx"

SEED = 52
TOTAL_ROWS = 9000
rng = np.random.default_rng(SEED)

## 2. Synthetic design principles

The synthetic DS1 generator is designed to provide the Paper I analysis-ready structure without the original proprietary values.

The generation logic introduces controlled dependencies among:

- alloy family,
- oven recipe,
- chemistry,
- measured tensile properties, and
- NaMo-like estimates

In [2]:
IMS_OPTIONS = ['6060', '6063', '6005 v1', '6005 v2', '6082 v1', '6082 v2', '6082 v3']
IMS_PROBABILITIES = np.array([0.16, 0.14, 0.15, 0.13, 0.15, 0.14, 0.13])
RECIPE_OPTIONS = [
    "Oven Recipe 1",
    "Oven Recipe 2",
    "Oven Recipe 3",
    "Oven Recipe 4",
    "Oven Recipe 5",
    "Oven Recipe 6",
]

STRENGTH_CODE = {
    '6060': 0.0,
    '6063': 0.3,
    '6005 v1': 0.8,
    '6005 v2': 1.1,
    '6082 v1': 1.5,
    '6082 v2': 1.7,
    '6082 v3': 2.0,
}

RECIPE_HEAT = {
    "Oven Recipe 1": 0.6,
    "Oven Recipe 2": 0.2,
    "Oven Recipe 3": -0.6,
    "Oven Recipe 4": 0.5,
    "Oven Recipe 5": -0.8,
    "Oven Recipe 6": 0.1,
}

RECIPE_STRENGTH = {
    "Oven Recipe 1": 18,
    "Oven Recipe 2": 8,
    "Oven Recipe 3": -10,
    "Oven Recipe 4": 15,
    "Oven Recipe 5": -16,
    "Oven Recipe 6": 4,
}

RECIPE_WEIGHTS = {
    '6060': np.array([0.10, 0.18, 0.22, 0.10, 0.28, 0.12]),
    '6063': np.array([0.14, 0.18, 0.14, 0.14, 0.20, 0.20]),
    '6005 v1': np.array([0.20, 0.24, 0.08, 0.18, 0.18, 0.12]),
    '6005 v2': np.array([0.22, 0.16, 0.10, 0.20, 0.18, 0.14]),
    '6082 v1': np.array([0.16, 0.14, 0.10, 0.22, 0.20, 0.18]),
    '6082 v2': np.array([0.18, 0.12, 0.08, 0.24, 0.18, 0.20]),
    '6082 v3': np.array([0.12, 0.12, 0.10, 0.20, 0.22, 0.24]),
}

def clip(values, low, high):
    return np.clip(values, low, high)


## 3. Generate chemistry, measured properties, and NaMo-like estimates

This section generates one synthetic row per sample.

For each row, the notebook assigns:

- an alloy label,
- an oven-recipe label,
- chemistry values,
- measured targets (`Rp02`, `Rm`), and
- corresponding synthetic NaMo estimates

The NaMo-like estimates are deliberately related to, but not identical with, the measured targets. This preserves a meaningful distinction between the mechanistic and data-driven representations used later in the Paper I analysis notebook.

In [3]:
# -------------------------------------------------------------------------
# Synthetic chemistry model
#
# The chemistry generation uses a small set of shared latent drivers:
#
#   strength : ordered alloy-family coordinate
#              (low for 6060/6063, high for 6005/6082 variants)
#
#   heat     : ordered recipe coordinate
#              (positive values represent more strengthening recipes,
#               negative values represent milder/softening recipes)
#
#   batch    : lot-to-lot variation shared across several variables
#
#   noise    : element-specific random scatter
#
# The purpose is not to mimic exact industrial chemistry limits, but to
# create a transparent synthetic dataset with plausible structure:
#
# - major alloying elements (Mg, Si) rise with alloy strength
# - some secondary elements (Mn, Cu, Cr, Zn, Zr) activate mainly in
#   stronger alloy families
# - trace elements (B, Ca, Ni, Ti, V) remain small and mostly random
# - Al is computed as the balance remainder rather than generated directly
#
# Each element is therefore generated from the same general logic:
#
#   value = base level
#         + alloy-family effect
#         + recipe effect (if relevant)
#         + lot effect (if relevant)
#         + local random variation
#
# followed by clipping to a plausible synthetic range.
# -------------------------------------------------------------------------


ims = rng.choice(IMS_OPTIONS, size=TOTAL_ROWS, p=IMS_PROBABILITIES)
recipes = np.array([rng.choice(RECIPE_OPTIONS, p=RECIPE_WEIGHTS[value]) for value in ims])
strength = pd.Series(ims).map(STRENGTH_CODE).to_numpy()
heat = pd.Series(recipes).map(RECIPE_HEAT).to_numpy()
batch = rng.normal(0, 1, TOTAL_ROWS)
process = rng.normal(0, 1, TOTAL_ROWS)

# Shared helper functions for transparent synthetic chemistry generation
def relu(x):
    return np.maximum(x, 0.0)

def make_major(base, alloy_slope=0.0, recipe_slope=0.0, batch_slope=0.0,
               noise_sd=0.0, low=0.0, high=1.0):
    """
    Major alloying element.

    Terms:
    - base: typical central level for the dataset
    - alloy_slope * strength: systematic increase across alloy families
    - recipe_slope * heat: optional dependence on recipe severity
    - batch_slope * batch: shared lot-to-lot variation
    - N(0, noise_sd): local random scatter
    """
    values = (
        base
        + alloy_slope * strength
        + recipe_slope * heat
        + batch_slope * batch
        + rng.normal(0, noise_sd, TOTAL_ROWS)
    )
    return clip(values, low, high)

def make_activated(base, threshold=0.0, alloy_slope=0.0, recipe_slope=0.0,
                   random_span=0.0, noise_sd=0.0, low=0.0, high=1.0):
    """
    Secondary / activated element.

    Terms:
    - base: low background level
    - alloy_slope * relu(strength - threshold): activates mainly in stronger alloys
    - recipe_slope * heat: optional recipe dependence
    - random_span * U(0,1): small positive random addition
    - N(0, noise_sd): local random scatter
    """
    values = (
        base
        + alloy_slope * relu(strength - threshold)
        + recipe_slope * heat
        + random_span * rng.random(TOTAL_ROWS)
        + rng.normal(0, noise_sd, TOTAL_ROWS)
    )
    return clip(values, low, high)

def make_trace(base, random_span=0.0, alloy_slope=0.0, threshold=0.0,
               noise_sd=0.0, low=0.0, high=1.0):
    """
    Trace / impurity-like element.

    Terms:
    - base: small background level
    - random_span * U(0,1): dominant random contribution
    - alloy_slope * relu(strength - threshold): optional weak enrichment in stronger alloys
    - N(0, noise_sd): local random scatter
    """
    values = (
        base
        + random_span * rng.random(TOTAL_ROWS)
        + alloy_slope * relu(strength - threshold)
        + rng.normal(0, noise_sd, TOTAL_ROWS)
    )
    return clip(values, low, high)


# Major alloying elements
Mg = make_major(
    base=0.34,          # baseline Mg level for the weakest alloy family
    alloy_slope=0.17,   # stronger alloys tend to contain more Mg
    batch_slope=0.04,   # shared batch-to-batch drift
    noise_sd=0.028,     # local row-level scatter
    low=0.30, high=1.15
)

Si = make_major(
    base=0.40,          # baseline Si level
    alloy_slope=0.14,   # stronger alloys tend to contain more Si
    recipe_slope=0.04,  # weak correlation with recipe severity
    noise_sd=0.030,
    low=0.30, high=1.10
)

# Secondary strengthening additions
Mn = make_activated(
    base=0.005,         # near-zero background
    threshold=0.4,      # starts increasing mainly for stronger alloy families
    alloy_slope=0.035,
    noise_sd=0.008,
    low=0.0, high=0.20
)

Cu = make_activated(
    base=0.004,
    threshold=0.7,
    alloy_slope=0.020,
    random_span=0.010,  # small independent positive variation
    noise_sd=0.004,
    low=0.0, high=0.14
)

Cr = make_activated(
    base=0.002,
    threshold=0.8,
    alloy_slope=0.010,
    noise_sd=0.003,
    low=0.0, high=0.05
)

Zn = make_activated(
    base=0.008,
    threshold=1.0,
    alloy_slope=0.010,
    random_span=0.012,
    noise_sd=0.003,
    low=0.0, high=0.08
)

Zr = make_activated(
    base=0.001,
    threshold=0.9,
    alloy_slope=0.006,
    noise_sd=0.0015,
    low=0.0, high=0.03
)

# Impurity / trace-like variables
Fe = make_trace(
    base=0.08,          # Fe acts here as a generic background impurity
    random_span=0.020,
    alloy_slope=0.012,  # slightly higher background in stronger alloy families
    threshold=0.2,
    noise_sd=0.009,
    low=0.03, high=0.25
)

B = make_trace(
    base=0.0010,
    random_span=0.0010,
    noise_sd=0.00025,
    low=0.0002, high=0.006
)

Ca = make_trace(
    base=0.0006,
    random_span=0.0018,
    noise_sd=0.0003,
    low=0.0001, high=0.008
)

Ni = make_trace(
    base=0.0008,
    random_span=0.0016,
    noise_sd=0.00025,
    low=0.0, high=0.008
)

Ti = make_trace(
    base=0.004,
    random_span=0.010,
    noise_sd=0.002,
    low=0.0, high=0.05
)

V = make_trace(
    base=0.002,
    random_span=0.010,
    noise_sd=0.002,
    low=0.0, high=0.05
)

# Al is not generated independently. It is computed as the balance remainder
# after all explicitly modelled alloying and trace elements have been assigned.
# A small "other elements / closure" term is subtracted so that Al behaves like
# the dominant balance component rather than summing exactly to 100 by construction.
other_elements = clip(rng.normal(0.14, 0.03, TOTAL_ROWS), 0.06, 0.24)

Al = clip(
    100 - (B + Ca + Cr + Cu + Fe + Mg + Mn + Ni + Si + Ti + V + Zn + Zr) - other_elements,
    97.6, 99.4
)

In [4]:
# -------------------------------------------------------------------------
# Synthetic tensile-property model
#
# The property generation is intentionally simple and transparent.
#
# First, a latent Rp0.2 signal is built from:
# - a constant baseline level
# - alloy-family strengthening (strength)
# - contributions from key chemistry variables (Mg, Si, Cu, Cr, Mn)
# - oven-recipe strengthening
# - shared lot/process variation
#
# Measured Rp02 is then obtained by adding random noise and clipping to a
# plausible range.
#
# Rm is built from Rp02 plus an additional margin representing the gap between
# yield strength and ultimate tensile strength. That margin is increased by:
# - stronger / hotter recipes
# - elevated Si
# - random scatter
#
# The NaMo-like estimates are generated from a similar but not identical logic:
# - they respond to the same broad drivers
# - they are smoother and somewhat less flexible than the measured targets
# - they therefore act as a synthetic mechanistic estimate rather than a copy
#
# This gives the public Paper I methodology a meaningful distinction between:
# - measured properties
# - NaMo-based estimates
# - purely data-driven regression
# -------------------------------------------------------------------------

# ----- Synthetic Rp0.2 signal -----
rp02_base = 220
rp02_alloy_term = 40 * strength                  # stronger alloy families have higher yield strength
rp02_mg_term = 54 * (Mg - 0.35)                 # Mg enrichment raises strength
rp02_si_term = 38 * (Si - 0.40)                 # Si enrichment raises strength
rp02_cu_term = 55 * Cu                          # Cu contributes strongly when present
rp02_cr_term = 28 * Cr                          # Cr contributes moderately
rp02_mn_term = 20 * Mn                          # Mn contributes moderately
rp02_recipe_term = pd.Series(recipes).map(RECIPE_STRENGTH).to_numpy()  # oven recipe effect
rp02_batch_term = 6 * batch                     # shared lot-to-lot variation
rp02_process_term = 4 * process                 # process-level variation not encoded in chemistry

signal = (
    rp02_base
    + rp02_alloy_term
    + rp02_mg_term
    + rp02_si_term
    + rp02_cu_term
    + rp02_cr_term
    + rp02_mn_term
    + rp02_recipe_term
    + rp02_batch_term
    + rp02_process_term
)

# Convert the latent signal to a measured synthetic Rp0.2 value
Rp02 = clip(
    signal + rng.normal(0, 6, TOTAL_ROWS),      # measurement / unmodelled scatter
    210, 380
)


# ----- Synthetic Rm signal -----
rm_margin_base = 18
rm_margin_recipe = 9 * np.maximum(heat, 0)      # only the "hotter/stronger" side of recipe severity adds margin
rm_margin_si = 30 * np.maximum(Si - 0.45, 0)    # elevated Si increases the Rp0.2-to-Rm gap
rm_margin_noise = rng.normal(0, 4, TOTAL_ROWS)

Rm = clip(
    Rp02 + rm_margin_base + rm_margin_recipe + rm_margin_si + rm_margin_noise,
    Rp02 + 12,                                  # enforce Rm > Rp0.2 by a minimum margin
    400
)


# ----- Replacement NaMo Rp0.2 estimate -----
namo_base = 228
namo_alloy_term = 26 * strength                 # smoother alloy-family trend than measured Rp0.2
namo_mg_term = 42 * (Mg - 0.35)
namo_si_term = 28 * (Si - 0.40)
namo_cu_term = 22 * Cu
namo_mn_term = 11 * Mn
namo_recipe_term = 10 * heat                    # milder recipe dependence than measured Rp0.2
namo_high_strength_penalty = -9 * np.maximum(strength - 1.0, 0)  # reduces overly aggressive growth in strongest alloys
namo_batch_term = 3 * batch                     # some lot-to-lot sensitivity retained
namo_low_strength_offset = np.where(strength < 0.4, 9, 0)        # slight boost in the low-strength regime

namo_signal = (
    namo_base
    + namo_alloy_term
    + namo_mg_term
    + namo_si_term
    + namo_cu_term
    + namo_mn_term
    + namo_recipe_term
    + namo_high_strength_penalty
    + namo_batch_term
    + namo_low_strength_offset
)

NaMo_Rp02 = clip(
    namo_signal + rng.normal(0, 11, TOTAL_ROWS),   # larger residual spread than the deterministic core
    200, 360
)

# ----- Replacement NaMo Rm estimate -----

NaMo_Rm = clip(
    NaMo_Rp02 + 16 + 7 * np.maximum(heat, 0) + rng.normal(0, 8, TOTAL_ROWS),
    NaMo_Rp02 + 8,
    395
)

## 4. Construct the ML-ready DS1 table

The categorical variables are retained both as readable string labels and as one-hot encoded columns.

In the final `ds1` below, the 27+2 column layout of DS1 corresponds to dropping 'IMS' and 'Oven Recipe'. After splitting of cross-validation folds, this is done in the Paper I analysis notebook.

In [5]:
base_df = pd.DataFrame({
    "IMS": ims.astype(str),
    "Oven Recipe": recipes.astype(str),
    "Al": np.round(Al, 4),
    "B": np.round(B, 5),
    "Ca": np.round(Ca, 5),
    "Cr": np.round(Cr, 5),
    "Cu": np.round(Cu, 5),
    "Fe": np.round(Fe, 5),
    "Mg": np.round(Mg, 5),
    "Mn": np.round(Mn, 5),
    "Ni": np.round(Ni, 5),
    "Si": np.round(Si, 5),
    "Ti": np.round(Ti, 5),
    "V": np.round(V, 5),
    "Zn": np.round(Zn, 5),
    "Zr": np.round(Zr, 5),
    "Rp02": np.round(Rp02, 1),
    "Rm": np.round(Rm, 1),
})
base_df.index = np.arange(100000, 100000 + len(base_df))

cat_df = pd.DataFrame({
    "IMS": pd.Categorical(base_df["IMS"], categories=IMS_OPTIONS, ordered=True),
    "Oven Recipe": pd.Categorical(base_df["Oven Recipe"], categories=RECIPE_OPTIONS, ordered=True),
}, index=base_df.index)

ohe_df = pd.get_dummies(
    cat_df,
    columns=["IMS", "Oven Recipe"],
    prefix=["IMS", "Oven Recipe"],
    dtype=int
)

namo_df = pd.DataFrame({
    "NaMo Rp02": np.round(NaMo_Rp02, 1),
    "NaMo Rm": np.round(NaMo_Rm, 1),
}, index=base_df.index)

ds1 = pd.concat([base_df, ohe_df], axis=1)

## 5. Save and inspect the generated data

In [6]:

print(f"Saved DS1 workbook: {DS1_FILE}")
print(f"Saved NaMo workbook: {NAMO_FILE}")
print(f"DS1 shape: {ds1.shape}")
print(f"NaMo shape: {namo_df.shape}")

print("\nDS1 columns:")
print(list(ds1.columns))

display(ds1.head())
display(namo_df.head())

ds1.to_excel(DS1_FILE)
namo_df.to_excel(NAMO_FILE)

Saved DS1 workbook: C:\Users\chrisdoi\OneDrive - NTNU\PhD\Thesis\codex\github\paper_I\data\DS1.xlsx
Saved NaMo workbook: C:\Users\chrisdoi\OneDrive - NTNU\PhD\Thesis\codex\github\paper_I\data\NaMo_syntetic.xlsx
DS1 shape: (9000, 31)
NaMo shape: (9000, 2)

DS1 columns:
['IMS', 'Oven Recipe', 'Al', 'B', 'Ca', 'Cr', 'Cu', 'Fe', 'Mg', 'Mn', 'Ni', 'Si', 'Ti', 'V', 'Zn', 'Zr', 'Rp02', 'Rm', 'IMS_6060', 'IMS_6063', 'IMS_6005 v1', 'IMS_6005 v2', 'IMS_6082 v1', 'IMS_6082 v2', 'IMS_6082 v3', 'Oven Recipe_Oven Recipe 1', 'Oven Recipe_Oven Recipe 2', 'Oven Recipe_Oven Recipe 3', 'Oven Recipe_Oven Recipe 4', 'Oven Recipe_Oven Recipe 5', 'Oven Recipe_Oven Recipe 6']


,IMS,Oven Recipe,Al,B,Ca,Cr,Cu,Fe,Mg,Mn,...,IMS_6005 v2,IMS_6082 v1,IMS_6082 v2,IMS_6082 v3,Oven Recipe_Oven Recipe 1,Oven Recipe_Oven Recipe 2,Oven Recipe_Oven Recipe 3,Oven Recipe_Oven Recipe 4,Oven Recipe_Oven Recipe 5,Oven Recipe_Oven Recipe 6
100000,6082 v1,Oven Recipe 5,98.3634,0.00117,0.00178,0.00610,0.02964,0.10159,0.66644,0.03827,...,0,1,0,0,0,0,0,0,1,0
100001,6005 v2,Oven Recipe 4,98.4711,0.00141,0.00219,0.00584,0.02362,0.10750,0.56027,0.03456,...,1,0,0,0,0,0,0,1,0,0
100002,6005 v2,Oven Recipe 6,98.6840,0.00125,0.00127,0.00269,0.01597,0.09469,0.46049,0.01592,...,1,0,0,0,0,0,0,0,0,1
100003,6060,Oven Recipe 1,98.9386,0.00099,0.00207,0.00308,0.00000,0.11189,0.30000,0.00692,...,0,0,0,0,1,0,0,0,0,0
100004,6005 v2,Oven Recipe 6,98.5566,0.00153,0.00218,0.00559,0.01998,0.11586,0.57184,0.01956,...,1,0,0,0,0,0,0,0,0,1


,NaMo Rp02,NaMo Rm
100000,273.6,287.1
100001,280.8,301.6
100002,258.6,268.9
100003,243.8,269.9
100004,281.1,296.0
